# 🔗 LangChain — Complete End-to-End Guide

**IIST Agentic AI Training Program · Days 8–9**

---

This notebook takes you from zero LangChain knowledge to building agents with tools, memory, and retrieval — all runnable in Google Colab.

### What We'll Cover

| # | Topic | What You'll Build |
|---|-------|-------------------|
| 1 | **Setup & First Call** | Raw OpenAI vs LangChain — same result, different ergonomics |
| 2 | **Prompt Templates** | Reusable, parameterized prompts |
| 3 | **LCEL (Pipe Operator)** | Composable chains: prompt \| model \| parser |
| 4 | **Output Parsers** | Structured JSON/Pydantic outputs |
| 5 | **Memory** | Conversational chains that remember context |
| 6 | **Tools** | Custom tools + built-in tools |
| 7 | **Agents (ReAct)** | LLM that autonomously picks and uses tools |
| 8 | **RAG Chain** | Retrieval-Augmented Generation with ChromaDB |
| 9 | **Routing & Parallel** | Conditional chains + parallel execution |
| 10 | **Streaming** | Real-time token-by-token output |

---

> **Prerequisites**: Basic Python. No prior LangChain experience needed.
>
> **API Key Required**: You'll need an OpenAI API key. Get one at [platform.openai.com](https://platform.openai.com)

---
## 1. Setup & Installation

We install the core LangChain packages. Notice how LangChain is modular — each provider (OpenAI, Anthropic, etc.) has its own package.

In [ ]:
# Install all packages we'll need throughout the notebook
!pip install -q \
    langchain \
    langchain-openai \
    langchain-community \
    langchain-core \
    chromadb \
    pydantic \
    tiktoken

In [ ]:
# Set your API key
# Option 1: Paste directly (for quick testing only — never commit this)
# Option 2: Use Colab Secrets (recommended)

import os

# --- PASTE YOUR KEY HERE ---
os.environ["OPENAI_API_KEY"] = "sk-..."  # Replace with your actual key

# --- OR use Colab Secrets (recommended) ---
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

---
## 2. Raw OpenAI vs LangChain — Side by Side

Let's see the **exact same task** done both ways. This answers the question: *"Why does LangChain exist?"*

In [ ]:
# ═══════════════════════════════════════
# METHOD 1: Raw OpenAI SDK
# ═══════════════════════════════════════
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is LangChain in one sentence?"}
    ]
)

print("[Raw OpenAI]")
print(response.choices[0].message.content)

In [ ]:
# ═══════════════════════════════════════
# METHOD 2: LangChain
# ═══════════════════════════════════════
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# Initialize model — same model, wrapped in LangChain's interface
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# The invoke() method is the universal LangChain interface
response = llm.invoke([
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What is LangChain in one sentence?")
])

print("[LangChain]")
print(response.content)
print(f"\nTokens used: {response.usage_metadata}")

### 💡 Key Takeaway

Same result, but LangChain gives you:
- **Unified interface** — swap OpenAI for Anthropic or Gemini by changing one line
- **Composability** — chain prompts, models, and parsers together (LCEL)
- **Built-in tooling** — memory, retrieval, agents, streaming
- **Observability** — plug in LangSmith for tracing

The real power shows up when you start *composing* things — which is what LCEL does.

---
## 3. Prompt Templates

Hardcoded strings don't scale. Prompt templates let you create reusable, parameterized prompts.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# ── Simple template with one variable ──
simple_prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple terms for a 10-year-old."
)

# Format it — this creates the actual messages
messages = simple_prompt.format_messages(topic="quantum computing")
print("Formatted messages:")
for msg in messages:
    print(f"  [{msg.type}]: {msg.content}")

In [ ]:
# ── Multi-role template (system + user) ──
expert_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {role}. Always give practical, actionable advice."),
    ("human", "{question}")
])

# Invoke directly with the LLM
response = llm.invoke(
    expert_prompt.format_messages(
        role="Python developer",
        question="What's the best way to handle errors in async Python code?"
    )
)
print(response.content)

---
## 4. LCEL — The Pipe Operator (This Changes Everything)

**LCEL** (LangChain Expression Language) lets you compose components using the `|` pipe operator.

Think of it like Unix pipes: `prompt | model | parser`

Data flows left → right. Each component transforms the data and passes it to the next.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# ═══════════════════════════════════════
# YOUR FIRST LCEL CHAIN
# ═══════════════════════════════════════
#
# Data flow:
#   {"topic": "..."}
#       ↓
#   prompt template (fills in the variables)
#       ↓
#   LLM (generates a response)
#       ↓
#   StrOutputParser (extracts just the text string)
#       ↓
#   "clean string output"

prompt = ChatPromptTemplate.from_template(
    "Give me 3 fun facts about {topic}. Keep each fact to one sentence."
)

# The magic: compose with |
chain = prompt | llm | StrOutputParser()

# Invoke the chain
result = chain.invoke({"topic": "the Moon"})
print(result)
print(f"\nReturn type: {type(result)}")  # It's a clean string now!

In [ ]:
# ═══════════════════════════════════════
# MULTI-STEP CHAIN (Research Pipeline)
# ═══════════════════════════════════════
# Step 1: Generate a topic summary
# Step 2: Take that summary and create quiz questions

from langchain_core.runnables import RunnablePassthrough

# Step 1: Summarize
summarize_prompt = ChatPromptTemplate.from_template(
    "Summarize the topic '{topic}' in 3 bullet points. Be concise."
)
summarize_chain = summarize_prompt | llm | StrOutputParser()

# Step 2: Generate quiz from the summary
quiz_prompt = ChatPromptTemplate.from_template(
    "Based on this summary, create 3 multiple-choice quiz questions:\n\n{summary}"
)
quiz_chain = quiz_prompt | llm | StrOutputParser()

# Compose them: summary feeds into quiz
full_pipeline = (
    summarize_chain  # Produces the summary string
    | (lambda summary: {"summary": summary})  # Wrap for next prompt
    | quiz_chain  # Generates quiz from summary
)

result = full_pipeline.invoke({"topic": "photosynthesis"})
print(result)

### 💡 Why LCEL Matters

Every component in the chain implements the **Runnable** interface:
- `.invoke()` — run once
- `.stream()` — stream tokens
- `.batch()` — run on multiple inputs
- `.ainvoke()` — async version

This means ANY chain you build automatically supports streaming, batching, and async — for free.

---
## 5. Output Parsers — Structured Data from LLMs

LLMs return text. But you often need **structured data** (JSON, objects). Output parsers solve this.

In [ ]:
# ═══════════════════════════════════════
# METHOD 1: Pydantic Output Parser
# ═══════════════════════════════════════
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# Define the structure you want
class MovieReview(BaseModel):
    title: str = Field(description="The movie title")
    rating: float = Field(description="Rating out of 10")
    pros: List[str] = Field(description="List of positive aspects")
    cons: List[str] = Field(description="List of negative aspects")
    one_line_verdict: str = Field(description="A one-line verdict")

parser = PydanticOutputParser(pydantic_object=MovieReview)

prompt = ChatPromptTemplate.from_template(
    "Review the movie '{movie}'.\n\n{format_instructions}"
)

# Build the chain
chain = prompt | llm | parser

# Invoke — the result is a Pydantic object, not a string!
review = chain.invoke({
    "movie": "Inception",
    "format_instructions": parser.get_format_instructions()
})

print(f"Title: {review.title}")
print(f"Rating: {review.rating}/10")
print(f"Pros: {review.pros}")
print(f"Cons: {review.cons}")
print(f"Verdict: {review.one_line_verdict}")
print(f"\nType: {type(review)}")  # It's a MovieReview object!

In [ ]:
# ═══════════════════════════════════════
# METHOD 2: with_structured_output (simpler!)
# ═══════════════════════════════════════
# This uses OpenAI's native JSON mode — more reliable

class CityInfo(BaseModel):
    """Information about a city."""
    name: str = Field(description="City name")
    country: str = Field(description="Country")
    population_millions: float = Field(description="Approximate population in millions")
    famous_for: List[str] = Field(description="What the city is famous for")

# Bind structured output directly to the model
structured_llm = llm.with_structured_output(CityInfo)

city = structured_llm.invoke("Tell me about Tokyo")

print(f"City: {city.name}, {city.country}")
print(f"Population: {city.population_millions}M")
print(f"Famous for: {', '.join(city.famous_for)}")

---
## 6. Memory — Conversational Chains

By default, LLMs are **stateless** — each call is independent. Memory fixes this.

In [ ]:
# ═══════════════════════════════════════
# CONVERSATION WITH MESSAGE HISTORY
# ═══════════════════════════════════════
# The modern LangChain way: manage history yourself
# (cleaner than the old ConversationBufferMemory)

from langchain_core.messages import AIMessage
from langchain_core.prompts import MessagesPlaceholder

# Prompt with a slot for chat history
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly tutor. Keep answers brief (2-3 sentences max)."),
    MessagesPlaceholder(variable_name="history"),  # Past messages go here
    ("human", "{input}")
])

chain = chat_prompt | llm | StrOutputParser()

# ── Simulate a multi-turn conversation ──
history = []

def chat(user_message):
    """Send a message and maintain history."""
    response = chain.invoke({"input": user_message, "history": history})

    # Add both messages to history
    history.append(HumanMessage(content=user_message))
    history.append(AIMessage(content=response))

    return response

# Turn 1
print("You: My name is Soham and I'm learning about agents.")
print(f"AI: {chat('My name is Soham and I am learning about agents.')}\n")

# Turn 2 — does it remember?
print("You: What's my name and what am I learning about?")
print(f"AI: {chat('Whats my name and what am I learning about?')}\n")

# Turn 3 — follow-up
print("You: Give me a simple definition of an agent.")
print(f"AI: {chat('Give me a simple definition of an agent.')}")

print(f"\n📊 History length: {len(history)} messages")

---
## 7. Tools — Giving LLMs Superpowers

An LLM alone can only generate text. **Tools** let it interact with the real world: search the web, do math, call APIs, read files.

This is the bridge between a *chatbot* and an *agent*.

In [ ]:
# ═══════════════════════════════════════
# DEFINING CUSTOM TOOLS
# ═══════════════════════════════════════
from langchain_core.tools import tool

# The @tool decorator turns any Python function into a LangChain tool
# The docstring becomes the tool's description (the LLM reads this!)

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Use Python math syntax.
    Examples: '2 + 3', '45 * 12', '2 ** 10', 'round(3.14159, 2)'"""
    try:
        import math
        result = eval(expression, {"__builtins__": {}, "math": math,
                                    "round": round, "abs": abs, "pow": pow})
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {e}"

@tool
def get_word_count(text: str) -> str:
    """Count the number of words in a given text."""
    count = len(text.split())
    return f"The text contains {count} words."

@tool
def lookup_capital(country: str) -> str:
    """Look up the capital city of a country."""
    capitals = {
        "india": "New Delhi",
        "france": "Paris",
        "japan": "Tokyo",
        "brazil": "Brasília",
        "australia": "Canberra",
        "germany": "Berlin",
        "usa": "Washington, D.C.",
        "united states": "Washington, D.C.",
    }
    result = capitals.get(country.lower())
    return f"The capital of {country} is {result}." if result else f"Capital not found for {country}."

# Inspect a tool
print(f"Tool name: {calculate.name}")
print(f"Tool description: {calculate.description}")
print(f"Tool schema: {calculate.args_schema.model_json_schema()}")

In [ ]:
# ═══════════════════════════════════════
# BIND TOOLS TO THE MODEL
# ═══════════════════════════════════════
# This tells the LLM: "Hey, you have these tools available."
# The LLM can then CHOOSE to use them.

tools = [calculate, get_word_count, lookup_capital]

# bind_tools adds the tool schemas to every API call
llm_with_tools = llm.bind_tools(tools)

# Ask something that requires a tool
response = llm_with_tools.invoke("What is 247 * 389?")

print("Content:", response.content)
print("\nTool calls:")
for tc in response.tool_calls:
    print(f"  Tool: {tc['name']}")
    print(f"  Args: {tc['args']}")

In [ ]:
# ═══════════════════════════════════════
# MANUAL TOOL EXECUTION
# ═══════════════════════════════════════
# Understanding what happens under the hood:
# 1. LLM decides to call a tool
# 2. WE execute the tool
# 3. We send the result back to the LLM
# 4. LLM formulates the final answer

from langchain_core.messages import ToolMessage

# Step 1: LLM decides to use a tool
ai_message = llm_with_tools.invoke("What's the capital of India?")
print("Step 1 — LLM wants to call:", ai_message.tool_calls)

# Step 2: Execute the tool ourselves
tool_call = ai_message.tool_calls[0]
tool_fn = {"calculate": calculate, "get_word_count": get_word_count, "lookup_capital": lookup_capital}
tool_result = tool_fn[tool_call["name"]].invoke(tool_call["args"])
print(f"\nStep 2 — Tool result: {tool_result}")

# Step 3: Send the result back
messages = [
    HumanMessage(content="What's the capital of India?"),
    ai_message,  # includes the tool_calls
    ToolMessage(content=tool_result, tool_call_id=tool_call["id"])
]

# Step 4: LLM gives final answer
final = llm_with_tools.invoke(messages)
print(f"\nStep 3 — Final answer: {final.content}")

### 💡 This 4-Step Loop IS the Agent

The manual process above is exactly what an **agent** automates:
1. LLM decides → tool call
2. Execute tool → get result
3. Feed result back → LLM decides again
4. Repeat until LLM gives a final text answer

The next section puts this loop on autopilot.

---
## 8. Agents — The ReAct Loop on Autopilot

An **agent** is just the tool-calling loop from above, automated. LangChain's `create_react_agent` (via LangGraph) handles the loop for you.

In [ ]:
!pip install -q langgraph

In [ ]:
# ═══════════════════════════════════════
# CREATE A REACT AGENT (LangGraph)
# ═══════════════════════════════════════
from langgraph.prebuilt import create_react_agent

# This one line creates an agent with:
# - The ReAct loop (think → act → observe → repeat)
# - Automatic tool execution
# - Built-in stopping condition

agent = create_react_agent(llm, tools)

# Test 1: Simple tool use
print("═" * 50)
print("Test 1: Math calculation")
print("═" * 50)
result = agent.invoke({"messages": [{"role": "user", "content": "What is 2^10 + 3^5?"}]})
# Print the final message
for msg in result["messages"]:
    if hasattr(msg, 'content') and msg.content:
        role = msg.type if hasattr(msg, 'type') else 'unknown'
        if role in ('human', 'ai'):
            print(f"[{role}]: {msg.content}")

In [ ]:
# Test 2: Multi-tool reasoning
print("═" * 50)
print("Test 2: Multi-tool query")
print("═" * 50)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "What's the capital of Japan? Also calculate 100 / 3 rounded to 2 decimals."
    }]
})

# Show the full trace — every step the agent took
print("\n--- Full Agent Trace ---")
for msg in result["messages"]:
    role = msg.type if hasattr(msg, 'type') else 'unknown'
    if role == 'human':
        print(f"\n🧑 Human: {msg.content}")
    elif role == 'ai':
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"\n🔧 Agent calls: {tc['name']}({tc['args']})")
        if msg.content:
            print(f"\n🤖 Agent: {msg.content}")
    elif role == 'tool':
        print(f"   📋 Tool result: {msg.content}")

In [ ]:
# Test 3: Query that doesn't need tools
# The agent should just answer directly without using any tools
print("═" * 50)
print("Test 3: No tools needed")
print("═" * 50)

result = agent.invoke({
    "messages": [{"role": "user", "content": "What is the ReAct pattern in AI?"}]
})

final_msg = result["messages"][-1]
print(f"Agent: {final_msg.content}")
print(f"\nTools used: {len([m for m in result['messages'] if m.type == 'tool'])}")

---
## 9. RAG Chain — Retrieval-Augmented Generation

RAG lets the LLM answer questions about **your data** — documents it was never trained on.

The pipeline:
1. **Split** documents into chunks
2. **Embed** chunks into vectors
3. **Store** in a vector database (ChromaDB)
4. **Retrieve** relevant chunks for a query
5. **Generate** an answer using the retrieved context

In [ ]:
# ═══════════════════════════════════════
# STEP 1: Create sample documents
# ═══════════════════════════════════════
# In production, these would be PDFs, web pages, database records, etc.

from langchain_core.documents import Document

documents = [
    Document(
        page_content="LangChain is a framework for building applications powered by large language models (LLMs). "
        "It provides tools for prompt management, chaining multiple LLM calls, and integrating with external data sources. "
        "LangChain was created by Harrison Chase and first released in October 2022.",
        metadata={"source": "langchain_overview", "topic": "framework"}
    ),
    Document(
        page_content="LCEL (LangChain Expression Language) is the modern way to build chains in LangChain. "
        "It uses the pipe operator (|) to compose prompts, models, and output parsers into declarative pipelines. "
        "LCEL chains automatically support streaming, batching, and async execution.",
        metadata={"source": "lcel_guide", "topic": "lcel"}
    ),
    Document(
        page_content="Agents in LangChain use the ReAct pattern: the LLM reasons about what to do, takes an action "
        "(like calling a tool), observes the result, and repeats until the task is complete. "
        "Agents are different from chains because they make dynamic decisions at runtime.",
        metadata={"source": "agents_guide", "topic": "agents"}
    ),
    Document(
        page_content="RAG (Retrieval-Augmented Generation) is a technique that grounds LLM responses in external knowledge. "
        "The pipeline involves chunking documents, creating embeddings, storing them in a vector database, "
        "and retrieving relevant chunks at query time to provide context to the LLM.",
        metadata={"source": "rag_guide", "topic": "rag"}
    ),
    Document(
        page_content="LangGraph is a library for building stateful, multi-actor applications with LLMs. "
        "It extends LangChain with state machines, persistent checkpoints, and human-in-the-loop capabilities. "
        "LangGraph is recommended for production-grade agents that need reliability and control flow.",
        metadata={"source": "langgraph_intro", "topic": "langgraph"}
    ),
    Document(
        page_content="Vector databases like ChromaDB, Pinecone, and Qdrant store document embeddings for fast similarity search. "
        "ChromaDB is open-source and runs locally, making it ideal for development and prototyping. "
        "Pinecone and Qdrant offer managed cloud solutions for production workloads with scaling capabilities.",
        metadata={"source": "vectordb_comparison", "topic": "vector_db"}
    ),
]

print(f"Created {len(documents)} documents")
for doc in documents:
    print(f"  - [{doc.metadata['topic']}] {doc.page_content[:60]}...")

In [ ]:
# ═══════════════════════════════════════
# STEP 2: Split → Embed → Store in ChromaDB
# ═══════════════════════════════════════
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Split documents into smaller chunks
# (Our docs are small, but this is essential for large documents)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,     # Max characters per chunk
    chunk_overlap=50,   # Overlap between chunks for context continuity
)
chunks = splitter.split_documents(documents)
print(f"Split {len(documents)} documents into {len(chunks)} chunks")

# Create embeddings and store in ChromaDB
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="langchain_guide"
)

print(f"Stored {vectorstore._collection.count()} vectors in ChromaDB")

In [ ]:
# ═══════════════════════════════════════
# STEP 3: Create a Retriever
# ═══════════════════════════════════════
# A retriever wraps the vector store and returns relevant documents

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # Return top 3 most relevant chunks
)

# Test the retriever
results = retriever.invoke("How do agents work?")
print(f"Found {len(results)} relevant chunks:\n")
for i, doc in enumerate(results, 1):
    print(f"  {i}. [{doc.metadata.get('topic', 'unknown')}] {doc.page_content[:100]}...\n")

In [ ]:
# ═══════════════════════════════════════
# STEP 4: Build the RAG Chain
# ═══════════════════════════════════════
from langchain_core.runnables import RunnablePassthrough

# RAG prompt template
rag_prompt = ChatPromptTemplate.from_template("""
Answer the question based ONLY on the following context. 
If the context doesn't contain the answer, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer:""")

# Helper: format retrieved docs into a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# The RAG chain!
# RunnablePassthrough passes the question through unchanged
# while the retriever fetches relevant context
rag_chain = (
    {
        "context": retriever | format_docs,     # Retrieve → format
        "question": RunnablePassthrough()         # Pass question through
    }
    | rag_prompt     # Fill the template
    | llm            # Generate answer
    | StrOutputParser()  # Extract text
)

# Test the RAG chain!
print("Q: What is LCEL and why does it matter?")
print(f"A: {rag_chain.invoke('What is LCEL and why does it matter?')}")
print()
print("Q: What's the difference between agents and chains?")
print(f"A: {rag_chain.invoke('Whats the difference between agents and chains?')}")
print()
print("Q: Which vector database should I use for development?")
print(f"A: {rag_chain.invoke('Which vector database should I use for development?')}")

---
## 10. Routing & Parallel Execution

Real apps need **branching** (route to different chains based on input) and **parallelism** (run multiple chains simultaneously).

In [ ]:
# ═══════════════════════════════════════
# PARALLEL EXECUTION with RunnableParallel
# ═══════════════════════════════════════
from langchain_core.runnables import RunnableParallel

# Three chains that run AT THE SAME TIME
summary_chain = (
    ChatPromptTemplate.from_template("Summarize this topic in 1 sentence: {topic}")
    | llm | StrOutputParser()
)

pros_chain = (
    ChatPromptTemplate.from_template("List 3 advantages of {topic}. Just bullet points.")
    | llm | StrOutputParser()
)

cons_chain = (
    ChatPromptTemplate.from_template("List 3 challenges or disadvantages of {topic}. Just bullet points.")
    | llm | StrOutputParser()
)

# Run all three in parallel!
parallel_chain = RunnableParallel(
    summary=summary_chain,
    pros=pros_chain,
    cons=cons_chain
)

result = parallel_chain.invoke({"topic": "using AI agents in production"})

print("📝 Summary:")
print(result["summary"])
print("\n✅ Pros:")
print(result["pros"])
print("\n⚠️ Cons:")
print(result["cons"])

In [ ]:
# ═══════════════════════════════════════
# CONDITIONAL ROUTING with RunnableBranch
# ═══════════════════════════════════════
from langchain_core.runnables import RunnableBranch

# Different chains for different types of questions
math_chain = (
    ChatPromptTemplate.from_template(
        "You are a math tutor. Solve this step by step: {input}"
    ) | llm | StrOutputParser()
)

code_chain = (
    ChatPromptTemplate.from_template(
        "You are a Python expert. Answer this coding question with a code example: {input}"
    ) | llm | StrOutputParser()
)

general_chain = (
    ChatPromptTemplate.from_template(
        "Answer this question helpfully: {input}"
    ) | llm | StrOutputParser()
)

# Route based on the input content
router = RunnableBranch(
    # (condition, chain) pairs — first match wins
    (lambda x: any(w in x["input"].lower() for w in ["calculate", "math", "solve", "equation"]), math_chain),
    (lambda x: any(w in x["input"].lower() for w in ["code", "python", "function", "program"]), code_chain),
    general_chain  # Default fallback
)

# Test routing
print("--- Math Route ---")
print(router.invoke({"input": "Solve: what is the derivative of x^3 + 2x?"}))
print("\n--- Code Route ---")
print(router.invoke({"input": "Write a Python function to reverse a linked list"}))
print("\n--- General Route ---")
print(router.invoke({"input": "What is the capital of France?"}))

---
## 11. Streaming — Real-Time Token Output

Users shouldn't wait for the full response. Streaming shows tokens as they're generated.

In [ ]:
# ═══════════════════════════════════════
# STREAMING OUTPUT
# ═══════════════════════════════════════
# Every LCEL chain supports .stream() for free!

stream_chain = (
    ChatPromptTemplate.from_template("Write a short poem about {topic}.")
    | llm
    | StrOutputParser()
)

print("Streaming output (token by token):")
print("-" * 40)

for chunk in stream_chain.stream({"topic": "coding at midnight"}):
    print(chunk, end="", flush=True)  # Print each token as it arrives

print("\n" + "-" * 40)
print("Done!")

---
## 12. Putting It All Together — Full Agent with RAG

Let's combine everything: an agent that can search our knowledge base, do calculations, and answer questions dynamically.

In [ ]:
# ═══════════════════════════════════════
# AGENT + RAG + TOOLS = FULL SYSTEM
# ═══════════════════════════════════════

@tool
def search_knowledge_base(query: str) -> str:
    """Search the LangChain knowledge base for information about
    LangChain, LCEL, agents, RAG, LangGraph, or vector databases.
    Use this when the user asks about any of these topics."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant information found in the knowledge base."
    return "\n\n".join(
        f"[{doc.metadata.get('source', 'unknown')}]: {doc.page_content}"
        for doc in docs
    )

@tool
def calculate_cost(prompt_tokens: int, completion_tokens: int, model: str = "gpt-4o-mini") -> str:
    """Calculate the cost of an LLM API call given token counts.
    Supports models: gpt-4o-mini, gpt-4o"""
    pricing = {
        "gpt-4o-mini": {"input": 0.15, "output": 0.60},   # per 1M tokens
        "gpt-4o": {"input": 2.50, "output": 10.00},
    }
    if model not in pricing:
        return f"Unknown model: {model}. Supported: {list(pricing.keys())}"
    p = pricing[model]
    input_cost = (prompt_tokens / 1_000_000) * p["input"]
    output_cost = (completion_tokens / 1_000_000) * p["output"]
    total = input_cost + output_cost
    return f"Cost for {model}: Input ${input_cost:.6f} + Output ${output_cost:.6f} = Total ${total:.6f}"

# Create the full agent
full_tools = [search_knowledge_base, calculate_cost, calculate]
full_agent = create_react_agent(llm, full_tools)

# Helper to run and display agent results
def ask_agent(question):
    print(f"\n{'═' * 60}")
    print(f"🧑 Question: {question}")
    print(f"{'═' * 60}")

    result = full_agent.invoke({"messages": [{"role": "user", "content": question}]})

    for msg in result["messages"]:
        if msg.type == 'ai' and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  🔧 Tool: {tc['name']}({tc['args']})")
        elif msg.type == 'tool':
            print(f"  📋 Result: {msg.content[:150]}{'...' if len(msg.content) > 150 else ''}")
        elif msg.type == 'ai' and msg.content:
            print(f"\n🤖 Answer: {msg.content}")

# Test the full agent!
ask_agent("What is LCEL and why should I use it?")
ask_agent("How much would it cost to process 50000 input tokens and 10000 output tokens with gpt-4o?")
ask_agent("What's the difference between LangChain and LangGraph?")

---
## 🎯 Summary: The LangChain Building Blocks

Here's how everything connects:

```
┌─────────────────────────────────────────────────────────┐
│                    YOUR APPLICATION                      │
├──────────┬──────────┬──────────┬──────────┬─────────────┤
│ Prompts  │  Models  │ Parsers  │  Tools   │   Memory    │
│          │          │          │          │             │
│ Template │ ChatGPT  │ String   │ @tool    │ Chat        │
│ Messages │ Claude   │ Pydantic │ Search   │ History     │
│ Few-shot │ Gemini   │ JSON     │ Custom   │ Vector      │
├──────────┴──────────┴──────────┴──────────┴─────────────┤
│                LCEL (Pipe Operator)                      │
│         prompt | model | parser = chain                  │
├─────────────────────────────────────────────────────────┤
│              AGENTS (ReAct Loop)                         │
│       Think → Act → Observe → Repeat                    │
├─────────────────────────────────────────────────────────┤
│              RAG (Retrieval)                             │
│     Chunk → Embed → Store → Retrieve → Generate         │
└─────────────────────────────────────────────────────────┘
```

### Key Concepts to Remember

| Concept | One-liner |
|---------|----------|
| **LCEL** | Compose chains with `\|` — everything supports stream/batch/async |
| **Prompt Templates** | Reusable, parameterized prompts with system/user roles |
| **Output Parsers** | Get structured data (Pydantic objects) from LLM text |
| **Tools** | Python functions the LLM can choose to call |
| **Agents** | Automated loop: LLM decides which tools to use |
| **RAG** | Ground LLM responses in your own data |
| **Memory** | Maintain context across conversation turns |
| **Routing** | Send different queries to different chains |
| **Parallel** | Run multiple chains simultaneously |

### What's Next?

- **LangGraph** — Build stateful agents with state machines, checkpoints, and human-in-the-loop (Days 10-11)
- **LangSmith** — Trace, debug, and evaluate your chains in production
- **Advanced RAG** — HyDE, rerankers, multi-query retrieval (Days 6-7)
- **Multi-Agent Systems** — CrewAI, AutoGen (Day 14)

---

**IIST Agentic AI Training Program**

Module 4: LangChain — Gentle Introduction (Days 8-9)

*Build agents that think. Deploy systems that scale. Ship AI that works.*